# Receipt Intelligence — Agentic Pipeline for Receipt Extraction

This notebook implements a multimodal **Extractor → Validator → Router** pipeline for automatically extracting and validating structured data from receipt photos, using exclusively local models via [Ollama](https://ollama.com/).

## Architecture

```
  Receipt image
       │
       ▼
┌─────────────────────┐
│  AGENT 1 (Vision)   │  gemma4 → reads the receipt, extracts structured JSON
│     Extractor       │
└─────────┬───────────┘
           │
           ▼
┌─────────────────────┐
│  AGENT 2 (Python)   │  sum of items vs declared total → math_error flag
│    Math Checker     │  (no LLM, 100% deterministic)
└─────────┬───────────┘
           │
           ▼
┌─────────────────────┐
│  AGENT 3 (Reasoning)│  deepseek-r1 → checks consistency, adds comment
│      Reviewer       │
└─────────┬───────────┘
           │
           ▼
    RunnableBranch
     /           \
  ✅ OK      ⚠️ NEEDS_REVIEW
  (save)    (requires review)
```

## Design Pattern

The implemented pattern is a variant of **Reflection/Validator** with conditional routing:
- **Agent 1** uses a multimodal LLM for perception (vision → structured text)
- **Agent 2** is a deterministic checker that does not waste LLM tokens on simple logic
- **Agent 3** uses a reasoning LLM for semantic validation
- **RunnableBranch** implements conditional routing based on the check result

## Prerequisites

```bash
pip install -r requirements.txt
ollama pull gemma4:e4b
ollama pull deepseek-r1:8b
```

## 1. Configuration

Edit these parameters before running the notebook. Keeping all configurable values at the top of the file is a good practice: it makes the notebook reusable without having to search for hard-coded paths buried in the middle of the code.

In [11]:
from pathlib import Path

# --- Edit here ---
IMAGE_PATH    = Path("./sample_receipts/receipt_01.jpg")  # path to the single receipt
VISION_MODEL  = "gemma4:e4b"      # vision model for extraction
REVIEW_MODEL  = "deepseek-r1:8b"  # reasoning model for review
MATH_TOLERANCE = 0.05             # tolerance in euros for the math error flag
OUTPUT_CSV    = Path("./output/receipts.csv")  # where to save batch results
SKIP_REVIEWER_IF_OK = True        # skip Agent 3 if math check passes (saves time and tokens)
# -----------------

print(f"Image path          : {IMAGE_PATH}")
print(f"Vision model        : {VISION_MODEL}")
print(f"Review model        : {REVIEW_MODEL}")
print(f"Math tolerance      : ±{MATH_TOLERANCE}€")
print(f"Skip reviewer if OK : {SKIP_REVIEWER_IF_OK}")

Image path          : sample_receipts/receipt_01.jpg
Vision model        : gemma4:e4b
Review model        : deepseek-r1:8b
Math tolerance      : ±0.05€
Skip reviewer if OK : True


## 2. Imports and LLM Initialization

Two distinct models with complementary roles:
- **`llm_vision`** (gemma4): multimodal model with image-reading capability. Temperature 0 for deterministic output.
- **`llm_review`** (deepseek-r1): model specialized in reasoning. Slower but more precise for logical verification.

In [ ]:
import base64
import json
import re
from typing import Optional, List

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage

try:
    llm_vision = ChatOllama(model=VISION_MODEL, temperature=0, reasoning=False)
    print(f"✓ Vision LLM '{VISION_MODEL}' initialized")
except Exception as e:
    print(f"✗ Error initializing Vision LLM: {e}")
    llm_vision = None

try:
    # reasoning=False is critical: without it deepseek-r1 enters "extended chain of thought"
    # mode and can take minutes on long inputs like a receipt JSON.
    llm_review = ChatOllama(model=REVIEW_MODEL, temperature=0, reasoning=False)
    print(f"✓ Review LLM '{REVIEW_MODEL}' initialized (reasoning=False)")
except Exception as e:
    print(f"✗ Error initializing Review LLM: {e}")
    llm_review = None

## 3. Pydantic Schema

We use **Pydantic** to define the schema for extracted data. This has three advantages over regex parsing:
1. **Automatic validation**: if a required field is missing or has the wrong type, we get a clear error.
2. **Type hints**: the notebook reads like documentation — you know exactly what to expect from the model.
3. **Integration with `JsonOutputParser`**: format instructions can be injected directly into the prompt.

In [ ]:
class ReceiptItem(BaseModel):
    name: str = Field(description="Item name as it appears on the receipt (abbreviations included)")
    quantity: float = Field(description="Quantity purchased")
    unit_price: float = Field(description="Price per unit; negative for discounts")

class Receipt(BaseModel):
    store: Optional[str] = Field(None, description="Store name")
    date: Optional[str] = Field(None, description="Receipt date (free format)")
    time: Optional[str] = Field(None, description="Receipt time")
    items: List[ReceiptItem] = Field(default_factory=list, description="List of purchased items")
    total: float = Field(description="Total declared on the receipt — most important field")
    taxes: Optional[float] = Field(None, description="VAT/taxes amount")
    payment_method: Optional[str] = Field(None, description="Payment method used")

parser = JsonOutputParser(pydantic_object=Receipt)
print("Receipt schema defined:")
print(json.dumps(Receipt.model_json_schema(), indent=2, ensure_ascii=False))

## 4. Agent 1 — Extractor (Vision LLM)

The first agent receives the image and produces structured JSON. The `extract_receipt` function follows the **state signature** used by all agents in the pipeline: it receives a `state` dictionary and returns a dictionary enriched with new keys.

This pattern — called **State Graph** or **Reducer** — is the same used by LangGraph and allows agents to be connected without direct coupling.

In [ ]:
EXTRACTOR_PROMPT = (
    "You are an assistant that extracts structured data from receipts. "
    "Analyze this receipt and return ONLY a valid JSON object. "
    "Expected format:\n{format_instructions}\n\n"
    "Important rules:\n"
    "- Use null for missing fields.\n"
    "- Duplicate items must be repeated as many times as they appear.\n"
    "- Discounts go as items with a negative unit_price.\n"
    "- Read numbers exactly as they appear, do not perform any calculations.\n"
    "- The 'total' field is the most important: copy it directly from the receipt.\n"
    "Reply with JSON only."
)

def encode_image(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def extract_receipt(state: dict) -> dict:
    image_path = Path(state["image_path"])
    image_data = encode_image(image_path)

    message = HumanMessage(content=[
        {
            "type": "text",
            "text": EXTRACTOR_PROMPT.format(
                format_instructions=parser.get_format_instructions()
            )
        },
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}
    ])

    response = llm_vision.invoke([message])

    # Parsing with double fallback: JsonOutputParser → regex → exception
    try:
        raw = parser.parse(response.content)
        receipt = Receipt(**raw) if isinstance(raw, dict) else raw
    except Exception:
        match = re.search(r'\{.*\}', response.content, re.DOTALL)
        if not match:
            raise ValueError(f"No JSON found in model response: {response.content[:300]}")
        receipt = Receipt(**json.loads(match.group()))

    print(f"[Agent 1] Extracted {len(receipt.items)} items — declared total: {receipt.total}€")
    return {**state, "receipt": receipt, "math_error": False, "review_comment": None}

print("Agent 1 (Extractor) defined.")

## 5. Agent 2 — Math Checker (Pure Python)

This agent uses no LLM: it is pure deterministic Python. The principle is **don't use an LLM when the logic is exactly computable** — it is faster, cheaper (zero tokens) and more reliable.

It computes the sum `Σ(quantity × unit_price)` for all items and compares it to the declared total. If the difference exceeds `MATH_TOLERANCE`, it sets `math_error=True` in the state — this flag will guide the final routing.

In [ ]:
def math_checker(state: dict) -> dict:
    receipt: Receipt = state["receipt"]

    computed_sum = round(
        sum(
            item.quantity * item.unit_price
            for item in receipt.items
            if item.quantity is not None and item.unit_price is not None
        ),
        2
    )
    declared_total = receipt.total
    difference = round(abs(computed_sum - declared_total), 2)
    math_error = difference > MATH_TOLERANCE

    verdict = "⚠️  ERROR" if math_error else "✅ OK"
    print(f"[Agent 2] Computed sum: {computed_sum}€ | Declared total: {declared_total}€ | Diff: {difference}€ → {verdict}")

    return {
        **state,
        "computed_sum": computed_sum,
        "math_error": math_error,
        "math_difference": difference,
    }

print("Agent 2 (Math Checker) defined.")

## 6. Agent 3 — Reviewer (Reasoning LLM, conditional)

The third agent uses a reasoning LLM for semantic validation: checks item consistency, identifies potential OCR errors, and produces a comment.

**Two key optimizations compared to a plain reviewer:**

1. **`reasoning=False`**: deepseek-r1 by default enters "extended chain of thought" mode and generates thousands of internal reasoning tokens before responding. With `reasoning=False`, the model responds directly — the difference can be minutes on long inputs.

2. **Conditional skip** (`SKIP_REVIEWER_IF_OK`): if Agent 2 has already verified that the math checks out, Agent 3 is skipped. The JSON is already consistent — calling an LLM to reconfirm it is wasteful. The reviewer fires **only when there is an error to investigate**, which is exactly when a reasoning model adds value.

In [ ]:
REVIEWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """\
You are a reviewer of data extracted from receipts via OCR.
You receive a raw JSON and must verify its consistency.

Rules:
1. Do not invent data: if a field is null, leave it null unless it is derivable from other fields.
2. Item names are OCR abbreviations: do NOT modify them.
3. Discounts appear immediately after the item they apply to.
4. Math error flagged: {math_error}. If True, try to identify which item might have the wrong value.
5. Add a 'review_comment' field with a brief explanation in English of what you verified.

Return ONLY a valid JSON object with the same fields as the original JSON plus 'review_comment'."""),
    ("user", """\
JSON to review:
{receipt_json}

Computed sum of items: {computed_sum}€
Declared total: {declared_total}€""")
])

def run_reviewer(state: dict) -> dict:
    # If the math check passes and SKIP_REVIEWER_IF_OK is True, skip the LLM call:
    # the JSON is already consistent and there is no point spending time/tokens to reconfirm it.
    if SKIP_REVIEWER_IF_OK and not state.get("math_error", False):
        print("[Agent 3] Skipped (math OK, SKIP_REVIEWER_IF_OK=True)")
        return {**state, "review_comment": "No review needed: math check passed."}

    receipt: Receipt = state["receipt"]
    receipt_dict = receipt.model_dump()

    response = llm_review.invoke(
        REVIEWER_PROMPT.format_messages(
            math_error=state.get("math_error", False),
            receipt_json=json.dumps(receipt_dict, indent=2, ensure_ascii=False),
            computed_sum=state.get("computed_sum", "N/A"),
            declared_total=receipt.total
        )
    )

    match = re.search(r'\{.*\}', response.content, re.DOTALL)
    if match:
        try:
            reviewed_raw = json.loads(match.group())
            review_comment = reviewed_raw.pop("review_comment", "No comment.")
            valid_fields = {k: v for k, v in reviewed_raw.items() if k in Receipt.model_fields}
            reviewed_receipt = Receipt(**{**receipt_dict, **valid_fields})
        except Exception as e:
            reviewed_receipt = receipt
            review_comment = f"Review failed ({e}), original data retained."
    else:
        reviewed_receipt = receipt
        review_comment = "Reviewer did not produce valid JSON."

    print(f"[Agent 3] Review complete. Comment: {review_comment[:80]}...")
    return {**state, "receipt": reviewed_receipt, "review_comment": review_comment}

print("Agent 3 (Reviewer) defined.")

## 7. Full Pipeline with RunnableBranch

We assemble the pipeline using **LCEL (LangChain Expression Language)** with the `|` operator. Each `RunnableLambda` wraps a Python function into an LCEL-compatible component.

The final `RunnableBranch` implements **conditional routing**:
- If `math_error=True` → status `NEEDS_REVIEW` (the receipt requires a human eye)
- Otherwise → status `OK` (the receipt is consistent and can be saved automatically)

This makes the pipeline **autonomous for simple cases** and **human-in-the-loop for ambiguous cases** — a fundamental principle of agentic systems in production.

In [ ]:
def _tag_ok(state: dict) -> dict:
    return {**state, "status": "OK"}

def _tag_needs_review(state: dict) -> dict:
    return {**state, "status": "NEEDS_REVIEW"}

routing_branch = RunnableBranch(
    (lambda state: state.get("math_error", False), RunnableLambda(_tag_needs_review)),
    RunnableLambda(_tag_ok)
)

receipt_pipeline = (
    RunnableLambda(extract_receipt)
    | RunnableLambda(math_checker)
    | RunnableLambda(run_reviewer)
    | routing_branch
)

print("Full pipeline assembled:")
print("  Agent 1 (Extractor) | Agent 2 (Math Checker) | Agent 3 (Reviewer) | RunnableBranch")

## 8. Test — Single Receipt

Run the pipeline on `IMAGE_PATH` defined at the top of the notebook. The result is a state dictionary with:
- `receipt`: validated Pydantic `Receipt` object
- `status`: `"OK"` or `"NEEDS_REVIEW"`
- `math_error`, `math_difference`: Math Checker results
- `review_comment`: LLM reviewer comment

In [ ]:
result = receipt_pipeline.invoke({"image_path": IMAGE_PATH})

print("\n" + "="*50)
print(f"STATUS         : {result['status']}")
print(f"Math error     : {result['math_error']} (difference: {result.get('math_difference', 0):.2f}€)")
print(f"Review comment : {result['review_comment']}")
print("="*50)
print("\n--- FINAL RECEIPT ---")
print(json.dumps(result["receipt"].model_dump(), indent=2, ensure_ascii=False))

## 9. Batch Processing and CSV Export

The `process_batch` function processes all images in a folder and saves the results to a CSV file. Each row corresponds to a receipt with the key fields and the status flag.

This transforms the pipeline from an interactive tool into an automatable batch system — for example, it could run weekly on a folder where the user drops receipt photos.

> **Note**: the `process_batch()` call below is commented out — uncomment it to run on the sample folder.

In [ ]:
import pandas as pd

def process_batch(image_dir: Path, output_csv: Path = OUTPUT_CSV) -> pd.DataFrame:
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    records = []

    image_files = sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg")) + list(image_dir.glob("*.png")))
    print(f"Found {len(image_files)} images in '{image_dir}'")

    for img_path in image_files:
        print(f"\nProcessing: {img_path.name} ...")
        try:
            res = receipt_pipeline.invoke({"image_path": img_path})
            rcpt: Receipt = res["receipt"]
            records.append({
                "file":             img_path.name,
                "store":            rcpt.store,
                "date":             rcpt.date,
                "time":             rcpt.time,
                "total":            rcpt.total,
                "taxes":            rcpt.taxes,
                "payment_method":   rcpt.payment_method,
                "n_items":          len(rcpt.items),
                "computed_sum":     res.get("computed_sum"),
                "math_difference":  res.get("math_difference", 0),
                "status":           res["status"],
                "review_comment":   res.get("review_comment", ""),
            })
        except Exception as e:
            print(f"  ERROR: {e}")
            records.append({"file": img_path.name, "status": "ERROR", "review_comment": str(e)})

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"\n✓ Results saved to: {output_csv}")
    return df

# Uncomment to run batch:
# df = process_batch(Path("./sample_receipts"))
# df